In [ ]:
# Python script for cleaning UAV orthomoasaic from Point Blue into polygon area
# STEP 1 resample
# Copyright (C) 2025 Alexandra Strang  and Gorden Jiang

import rioxarray
import geopandas
import xarray as xr
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

In [ ]:
# read in raster file
Crozier_UAV_raster = rioxarray.open_rasterio("Crozier_20201129_3031.tif",
                                             chunks = "auto")

# raster info
print(Crozier_UAV_raster)
print(f"CRS of raster: {Crozier_UAV_raster.rio.crs}\n")

<xarray.DataArray (band: 4, y: 199400, x: 208383)> Size: 166GB
dask.array<open_rasterio-6114b9fd3ed5f04525fda46b76cdc6f2<this-array>, shape=(4, 199400, 208383), dtype=uint8, chunksize=(1, 11520, 11520), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int32 16B 1 2 3 4
  * x            (x) float64 2MB 2.538e+05 2.538e+05 ... 2.564e+05 2.564e+05
  * y            (y) float64 2MB -1.343e+06 -1.343e+06 ... -1.345e+06 -1.345e+06
    spatial_ref  int32 4B 0
Attributes: (12/13)
    DataType:                Generic
    AREA_OR_POINT:           Area
    RepresentationType:      ATHEMATIC
    STATISTICS_COVARIANCES:  12443.36535428216,12440.02451445284,12378.164018...
    STATISTICS_MAXIMUM:      255
    STATISTICS_MEAN:         171.69533691088
    ...                      ...
    STATISTICS_MINIMUM:      0
    STATISTICS_SKIPFACTORX:  1
    STATISTICS_SKIPFACTORY:  1
    STATISTICS_STDDEV:       111.54983350181
    scale_factor:            1.0
    add_offset:              0.0
CRS o

In [ ]:
raster_band = Crozier_UAV_raster[0]

In [16]:
from rasterio.enums import Resampling

# --- create the Dask-backed 'reclassified_raster' ---

# 1. Define a downsampling factor.
# For example, a factor of 1000 will make the new pixels 1000x larger.
downscale_factor = 1000

# 2. Calculate the new dimensions of the downsampled raster.
new_width = int(raster_band.rio.width / downscale_factor)
new_height = int(raster_band.rio.height / downscale_factor)

# 3. Resample the raster to the new, lower resolution.
# This operation is also lazy with Dask. We use 'nearest' neighbor resampling
downsampled_raster = raster_band.rio.reproject(
    raster_band.rio.crs,
    shape=(new_height, new_width),
    resampling=Resampling.nearest
)


In [ ]:
# downsampled_raster.rio.to_raster("downsampled_raster.tif", dtype="uint8") # Use a memory-efficient data type